# 02 — Model Training Demo

This notebook demonstrates training the CognitiveTwinFusionModel on synthetic data,
visualises training curves, and runs `declare_state()` on a test batch.

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import matplotlib.pyplot as plt

from models.multimodal_fusion import CognitiveTwinFusionModel
from training.losses import CognitiveTwinLoss

torch.manual_seed(42)
np.random.seed(42)
print('Setup complete.')

## 1. Create Synthetic Dataset

In [ ]:
N = 64        # samples
N_CH = 4      # EEG channels
N_FREQS = 16  # scalogram freq bins
N_TIME = 64   # scalogram time steps

eeg_data = torch.randn(N, N_CH, N_FREQS, N_TIME)
eye_data = torch.randn(N, 64, 7)   # 64 timesteps, 7 features
hrv_data = torch.randn(N, 8, 10)   # 8 windows, 10 HRV features
labels   = torch.randint(0, 4, (N,))
av_targets = torch.rand(N, 2) * 2 - 1   # range [-1, 1]

dataset = TensorDataset(eeg_data, eye_data, hrv_data, labels, av_targets)
loader  = DataLoader(dataset, batch_size=16, shuffle=True)
print(f'Dataset: {N} samples, {len(loader)} batches')

## 2. Initialise Model & Loss

In [ ]:
model = CognitiveTwinFusionModel(
    n_channels_eeg=N_CH, n_freqs=N_FREQS, n_time=N_TIME,
    n_eye_features=7, n_hrv_features=10, n_classes=4
)
criterion = CognitiveTwinLoss()
optimizer = torch.optim.AdamW(
    list(model.parameters()) + list(criterion.parameters()), lr=1e-3
)
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Model parameters: {n_params:,}')

## 3. Training Loop

In [ ]:
N_EPOCHS = 20
history = {'total': [], 'cls': [], 'reg': [], 'aux': [], 'acc': []}

for epoch in range(1, N_EPOCHS + 1):
    model.train()
    epoch_loss, correct, total = 0.0, 0, 0
    for eeg_b, eye_b, hrv_b, lbl_b, av_b in loader:
        optimizer.zero_grad()
        outputs = model(eeg_b, eye_b, hrv_b)
        loss_dict = criterion(outputs, lbl_b, av_b)
        loss_dict['total'].backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        epoch_loss += loss_dict['total'].item() * len(lbl_b)
        correct += (outputs['final_probs'].argmax(1) == lbl_b).sum().item()
        total += len(lbl_b)
    acc = correct / total
    history['total'].append(epoch_loss / total)
    history['acc'].append(acc)
    if epoch % 5 == 0:
        print(f'Epoch {epoch:3d}/{N_EPOCHS}  loss={epoch_loss/total:.4f}  acc={acc:.3f}')

print('Training complete.')

## 4. Training Curves

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(history['total'], color='#58a6ff', lw=2)
ax1.set_title('Total Loss', fontsize=12)
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')

ax2.plot(history['acc'], color='#3fb950', lw=2)
ax2.axhline(0.25, color='#f78166', linestyle='--', label='Random baseline')
ax2.set_title('Accuracy', fontsize=12)
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy')
ax2.legend()

for ax in [ax1, ax2]:
    ax.set_facecolor('#0d1117')
    ax.tick_params(colors='#8b949e')
    ax.spines[:].set_color('#30363d')
fig.patch.set_facecolor('#0d1117')
plt.tight_layout()
plt.show()

## 5. declare_state() Demo

In [ ]:
import json

model.eval()
test_eeg = torch.randn(2, N_CH, N_FREQS, N_TIME)
test_eye = torch.randn(2, 64, 7)
test_hrv = torch.randn(2, 8, 10)

with torch.no_grad():
    outputs = model(test_eeg, test_eye, test_hrv)

states = model.declare_state(outputs)
for i, s in enumerate(states):
    print(f'Sample {i}:')
    print(json.dumps(s, indent=2))
    print()